# R3-02 · IA generativa y LLMs — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea R3 · *Especialista en IA Aplicada* · Semana 3 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Escribir **prompts** con rol, tarea, reglas y fuente para **resumir en lenguaje claro**.
2. Extraer **requisitos como JSON** y hacerlo cumplir con un **parser** y un **validador** propios.
3. Clasificar con **few-shot** y aceptar solo salidas válidas.
4. Entender **temperatura y alucinación** y **verificar cifras contra la fuente**.
5. Proteger datos personales con un **guardrail** y aplicar **verificación humana**.

**Competencia de salida:** usar un LLM sobre texto del Estado de forma verificable y responsable: el código no confía en la respuesta, la valida.

**Dato real:** fichas de ChileAtiende — Aumento de la PGU y Bono de Invierno (texto oficial vigente).

**Entregable:** que las **6 celdas de chequeo** muestren ✅.

In [ ]:
# ── Preparación del entorno (ejecuta esta celda primero) ──────────────────────
import os, urllib.request
import pandas as pd
import json, re

# "En vivo o caché": usa el archivo local; si falta (ej. Colab), lo baja del repo.
CSV = "fichas.csv"
if not os.path.exists(CSV):
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/fichas.csv", CSV)

df = pd.read_csv(CSV)
print("Datos cargados:", df.shape, "filas x columnas")
df.head()

In [ ]:
# Las dos fichas con que trabajaremos
pgu = df[df["id"] == 130457].iloc[0]        # Aumento de la PGU
invierno = df[df["id"] == 39484].iloc[0]    # Bono de Invierno

# Cada ficha tiene una introducción y luego secciones que empiezan con "## ".
def secciones(texto):
    return re.findall(r"^## (.+)$", texto, flags=re.M)

# Para extraer datos del Bono de Invierno usamos su introducción: lo que va antes de la primera sección.
fragmento = invierno["texto"].split("\n## ")[0].strip()

for f in [pgu, invierno]:
    print(f["titulo"], "·", f["institucion"], "·", f["n_palabras"], "palabras")
    print("   secciones:", secciones(f["texto"]))
print()
print(pgu["texto"][:700], "...")

## 1. ¿Qué es un LLM y qué cambia respecto de R3-01?

En R3-01 el modelo **contaba palabras**: no sabía que «PGU» y «Pensión Garantizada Universal» son
lo mismo. Un **LLM** (*Large Language Model*: Gemini, ChatGPT, Claude) se entrena con una tarea
engañosamente simple — **predecir la siguiente palabra** — sobre cantidades enormes de texto, y
para hacerlo bien termina capturando significado, sinónimos y contexto. Por eso puede **resumir,
reescribir, extraer y clasificar** sin que nadie lo programe para cada tarea.

Tú lo diriges con el **prompt** (la instrucción). Tres reglas que rinden de inmediato:

1. **Sé específico:** qué quieres, para quién y en qué formato («resume en lenguaje claro, máximo 60 palabras»).
2. **Da contexto:** pega el texto fuente, las opciones válidas, ejemplos.
3. **Acota la salida:** si quieres JSON, dilo, define las claves y prohíbe el texto extra.

Y una regla de oro para programar con LLMs: **el modelo es probabilístico** — el mismo prompt
puede dar respuestas distintas, y a veces **inventa** con total seguridad (*alucina*). Por eso,
en este módulo, **tu código no confía en la respuesta: la verifica**. Los chequeos revisan tus
funciones (el prompt, el parser, el validador, el verificador), nunca el texto exacto del LLM.

## 2. Tu clave gratuita de Gemini (opcional) y la regla de privacidad

Usaremos **Gemini** a través de **Google AI Studio**: gratis, sin tarjeta, con tu cuenta de Google.

1. Entra a **https://aistudio.google.com/apikey** e inicia sesión.
2. Pulsa **Create API key** y copia la clave.
3. En Colab: panel izquierdo → 🔑 **Secretos** → **Agregar un secreto nuevo** → nombre
   `GEMINI_API_KEY`, pega el valor y activa **Acceso del notebook**.

> 🔒 **Nunca pegues la clave en una celda**: quedaría guardada en el archivo y en cualquier copia
> que compartas. Úsala siempre desde los Secretos.

> 🛡️ **No envíes datos personales al LLM.** En la capa gratuita, el proveedor puede usar tus textos
> para mejorar sus modelos. Aquí trabajamos con fichas **públicas** de ChileAtiende; nunca con RUT,
> nombres de personas, datos de salud ni información reservada (lo automatizamos en el ejercicio 6).

**¿Sin clave?** No pasa nada: `preguntar_llm()` devuelve una **respuesta de respaldo** y todo el
notebook corre igual. La celda siguiente define `preguntar_llm(prompt, respaldo=...)`: es la única
puerta al modelo (cambiar de proveedor = cambiar solo esa celda). Colab ya trae el paquete
`google-genai`; si en otro entorno ves *No module named google*, ejecuta `!pip install -q google-genai`.

In [ ]:
# ── preguntar_llm(): el LLM (Gemini) detrás de UNA función ─────────────────────
# Cambiar de proveedor o de modelo = cambiar solo esta celda.
# En Colab: 🔑 (Secretos) → agrega GEMINI_API_KEY (gratis en aistudio.google.com).
import os

MODELO = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash-lite")

def _clave_gemini():
    try:                                   # Colab: secreto del notebook
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except Exception:                      # local: variable de entorno
        return os.environ.get("GEMINI_API_KEY")

def preguntar_llm(prompt, respaldo="[respuesta de respaldo — configura GEMINI_API_KEY para usar el LLM en vivo]"):
    """Envía el prompt a Gemini y devuelve el texto. Sin clave o sin red, devuelve
    `respaldo` para que el notebook siga corriendo (y se pueda verificar en CI)."""
    clave = _clave_gemini()
    if not clave:
        return respaldo
    try:
        from google import genai
        cliente = genai.Client(api_key=clave)
        return cliente.models.generate_content(model=MODELO, contents=prompt).text
    except Exception as e:
        print("⚠️ No se pudo llamar al LLM:", e)
        return respaldo

def get_llm():
    """Compatibilidad: devuelve una función generate(prompt)->str."""
    return preguntar_llm

In [ ]:
if _clave_gemini():
    print("🔑 Clave detectada → modo EN VIVO con", MODELO)
else:
    print("ℹ️ Sin GEMINI_API_KEY → modo RESPALDO: verás respuestas de ejemplo y todo funciona igual.")

print(preguntar_llm("Explica en una frase qué es ChileAtiende.",
                    respaldo="ChileAtiende es la red del Estado que orienta a las personas sobre trámites y beneficios públicos."))

### Las respuestas de respaldo

Sin clave, `preguntar_llm()` devuelve el `respaldo` que le pases. Abajo están los de este
módulo: **respuestas de ejemplo escritas a mano a partir del texto de las fichas** (no generadas
en vivo), con el formato típico de un LLM — incluido el JSON envuelto en ```` ```json ```` con una
frase antes. `RESPUESTA_DUDOSA` es una extracción con errores plantados, como las que produce un
modelo cuando alucina: la usaremos para probar que tu verificador los atrapa.

In [ ]:
RESPALDO_RESUMEN = ("Con la Reforma de Pensiones, la Pensión Garantizada Universal (PGU) subió. "
    "Desde febrero de 2026, si tienes 82 años o más recibes hasta $250.275, y si tienes entre 65 y 81 años, "
    "hasta $231.732. Desde septiembre de 2026, las personas de 75 años o más podrán recibir el monto máximo. "
    "Consulta con tu RUN cuándo aumentará tu PGU.")

RESPALDO_EXTRACCION = """Aquí está el JSON solicitado:
```json
{
  "tramite": "Bono de Invierno",
  "institucion": "Instituto de Previsión Social",
  "requisitos": [
    "Tener 65 años o más al 1 de mayo de 2026",
    "Recibir una pensión que no supere los $231.440"
  ],
  "montos_clp": [81257, 231440]
}
```"""

RESPUESTA_DUDOSA = """```json
{
  "tramite": "Bono de Invierno",
  "institucion": "Instituto de Previsión Social",
  "requisitos": [
    "Tener 65 años o más al 1 de mayo de 2026",
    "Postular en tu AFP antes del 30 de abril"
  ],
  "montos_clp": [85000, 231440]
}
```"""

RESPALDO_TEMA = "bonos-y-beneficios-del-estado"
print("Respaldos listos.")

## 3. Un prompt para resumir en lenguaje claro

La ficha de la PGU tiene 893 palabras repartidas en una introducción y 4 secciones, con siglas (PBSI, APSI, DIPRECA) y frases largas. Escribe `prompt_resumen(ficha, max_palabras=60)`, que arme un prompt con cuatro partes: **rol** (ya está), **tarea** (resumir la ficha — con su título — en *lenguaje claro*, en máximo `max_palabras` palabras), **reglas** (ya están) y la **fuente**: el texto completo precedido de `TEXTO DE LA FICHA:`.

### ✍️ Ejercicio 1 — Escribe `prompt_resumen`

In [ ]:
def prompt_resumen(ficha, max_palabras=60):
    rol = "Eres un funcionario de ChileAtiende que explica trámites a la ciudadanía."
    tarea = f"Resume la ficha «{ficha['titulo']}» en lenguaje claro, en máximo {max_palabras} palabras."
    reglas = ("Usa solo la información del texto; si un dato no está, no lo inventes. "
              "Conserva montos, edades y fechas tal como aparecen.")
    fuente = "TEXTO DE LA FICHA:\n" + ficha["texto"]
    return "\n".join([rol, tarea, reglas, "", fuente])

print(prompt_resumen(pgu)[:400], "...")

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    _p = prompt_resumen(pgu)
    assert pgu["titulo"] in _p, "el prompt debe nombrar la ficha"
    assert _p.count(pgu["texto"]) == 1, "el texto completo de la ficha debe ir una vez"
    assert "lenguaje claro" in _p.lower()
    assert "60" in _p and "45" in prompt_resumen(pgu, 45), "el límite debe salir de max_palabras"
    assert "TEXTO DE LA FICHA:" in _p
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'tarea = f"Resume la ficha «{ficha[\'titulo\']}» en lenguaje claro, en máximo {max_palabras} palabras."; fuente = "TEXTO DE LA FICHA:\\n" + ficha["texto"].')
    print("   Detalle:", e)

In [ ]:
resumen = preguntar_llm(prompt_resumen(pgu), respaldo=RESPALDO_RESUMEN)
print(resumen)
print(f"\n→ {len(resumen.split())} palabras (la ficha tiene {pgu['n_palabras']})")

## Extraer requisitos como datos estructurados

Una de las tareas más útiles de un LLM en el Estado: convertir **texto libre** en **datos con
forma** — un JSON con claves fijas que después se guarda en una tabla. El prompt de abajo define
el **contrato**: qué claves, de qué tipo, y la prohibición de inventar. Ejecútalo para ver la
respuesta (en vivo o de respaldo): fíjate que el LLM suele **envolver** el JSON en ```` ``` ```` y
agregar una frase antes. `json.loads` sobre esa respuesta cruda **falla**: necesitas un parser.

In [ ]:
def prompt_extraccion(ficha, fragmento):
    return f"""Extrae del texto los datos del trámite «{ficha['titulo']}» ({ficha['institucion']}).
Responde SOLO con un JSON con exactamente estas claves:
- "tramite": nombre del trámite (texto)
- "institucion": institución responsable (texto)
- "requisitos": lista de requisitos, cada uno una frase corta (lista de textos)
- "montos_clp": montos en pesos que aparezcan en el texto, como enteros sin $ ni puntos (lista de enteros)
No agregues requisitos ni montos que no estén en el texto.

TEXTO:
{fragmento}"""

respuesta_cruda = preguntar_llm(prompt_extraccion(invierno, fragmento), respaldo=RESPALDO_EXTRACCION)
print(respuesta_cruda)
try:
    json.loads(respuesta_cruda)
except json.JSONDecodeError as e:
    print("\n⚠️ json.loads directo falla:", e)

## 4. Un parser que no confía en el formato

Escribe `parsear_json(respuesta)`: busca la **primera** `{` y la **última** `}` del texto, intenta `json.loads` sobre ese trozo y devuelve el `dict`; si no hay llaves o el JSON es inválido, devuelve `None` (sin lanzar error). Así sobrevive a los ```` ``` ```` y a las frases de cortesía del modelo.

### ✍️ Ejercicio 2 — Escribe `parsear_json`

In [ ]:
def parsear_json(respuesta):
    texto = respuesta.strip()
    inicio = texto.find("{")
    fin = texto.rfind("}")
    if inicio == -1 or fin == -1 or fin < inicio:
        return None
    try:
        return json.loads(texto[inicio:fin + 1])
    except json.JSONDecodeError:
        return None

extraccion = parsear_json(respuesta_cruda)
print(extraccion)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    _d = parsear_json(RESPALDO_EXTRACCION)
    assert isinstance(_d, dict) and _d["tramite"] == "Bono de Invierno"
    assert parsear_json("Lo siento, no puedo ayudarte con eso.") is None
    assert parsear_json('{"a": 1,}') is None, "un JSON inválido debe dar None, no un error"
    assert parsear_json('Aquí va: {"a": [1, 2]} ¡listo!') == {"a": [1, 2]}
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "inicio = texto.find('{'); fin = texto.rfind('}'); return json.loads(texto[inicio:fin + 1]).")
    print("   Detalle:", e)

## 5. Validar el JSON contra el contrato

Que el JSON se pueda leer no significa que cumpla el contrato del prompt. Escribe `validar_extraccion(d)`, que devuelva una **lista de problemas** (vacía si todo está bien): claves que faltan, claves con el tipo equivocado, y montos que no sean enteros.

### ✍️ Ejercicio 3 — Escribe `validar_extraccion`

In [ ]:
CONTRATO = {"tramite": str, "institucion": str, "requisitos": list, "montos_clp": list}

def validar_extraccion(d):
    if not isinstance(d, dict):
        return ["la respuesta no es un objeto JSON"]
    problemas = []
    for clave, tipo in CONTRATO.items():
        if clave not in d:
            problemas.append(f"falta la clave '{clave}'")
        elif not isinstance(d[clave], tipo):
            problemas.append(f"'{clave}' debería ser {tipo.__name__}")
    if isinstance(d.get("requisitos"), list) and len(d["requisitos"]) == 0:
        problemas.append("la lista de requisitos está vacía")
    if isinstance(d.get("montos_clp"), list) and not all(isinstance(m, int) for m in d["montos_clp"]):
        problemas.append("los montos deben ser enteros (sin $ ni puntos)")
    return problemas

print("respaldo:", validar_extraccion(parsear_json(RESPALDO_EXTRACCION)))
print("malo:    ", validar_extraccion({"tramite": "Bono", "requisitos": "tener 65 años", "montos_clp": ["$81.257"]}))

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert validar_extraccion(parsear_json(RESPALDO_EXTRACCION)) == []
    _malo = validar_extraccion({"tramite": "Bono", "requisitos": "tener 65 años", "montos_clp": ["$81.257"]})
    assert isinstance(_malo, list) and len(_malo) == 3, "esperaba 3 problemas: falta institucion, requisitos no es lista, montos no enteros"
    assert len(validar_extraccion(None)) == 1
    assert validar_extraccion({"tramite": "x", "institucion": "y", "requisitos": [], "montos_clp": []}) == ["la lista de requisitos está vacía"]
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "Dentro del for: if clave not in d: problemas.append(...) / elif not isinstance(d[clave], tipo): problemas.append(...). Luego: if isinstance(d.get('montos_clp'), list) and not all(isinstance(m, int) for m in d['montos_clp']): problemas.append(...).")
    print("   Detalle:", e)

## 6. Few-shot: enseñar con ejemplos dentro del prompt

En R3-01 entrenaste un clasificador con 183 fichas. Con un LLM basta mostrar **unos pocos ejemplos** en el propio prompt (*few-shot*) y la lista de temas válidos. Completa `prompt_few_shot(titulo, ejemplos, temas)`: agrega cada ejemplo como dos líneas `Trámite: …` / `Tema: …`. El prompt termina en `Tema:` para que el modelo solo complete la etiqueta.

### ✍️ Ejercicio 4 — Escribe `prompt_few_shot`

In [ ]:
TEMAS = sorted(df["tema"].unique())
EJEMPLOS = [
    ("Bono Logro Escolar", "educacion"),
    ("Certificado de nacimiento para chilenos y chilenas en el extranjero", "chilenos-en-el-exterior"),
    ("Anticipo de primera pensión de CAPREDENA", "jubilacion"),
]

def prompt_few_shot(titulo, ejemplos, temas):
    lineas = ["Clasifica el trámite en UNO de estos temas (responde solo el tema):",
              ", ".join(temas), "", "Ejemplos:"]
    for tramite, tema in ejemplos:
        lineas.append(f"Trámite: {tramite}")
        lineas.append(f"Tema: {tema}")
    lineas += ["", f"Trámite: {titulo}", "Tema:"]
    return "\n".join(lineas)

print(prompt_few_shot("Aguinaldo de Navidad para pensionados IPS", EJEMPLOS, TEMAS))

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    _p = prompt_few_shot("Aguinaldo de Navidad para pensionados IPS", EJEMPLOS, TEMAS)
    for _t, _tema in EJEMPLOS:
        assert f"Trámite: {_t}\nTema: {_tema}" in _p, f"falta el ejemplo «{_t}»"
    assert all(t in _p for t in TEMAS), "el prompt debe listar los temas válidos"
    assert _p.rstrip().endswith("Trámite: Aguinaldo de Navidad para pensionados IPS\nTema:")
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'Dentro del for: lineas.append(f"Trámite: {tramite}") y lineas.append(f"Tema: {tema}").')
    print("   Detalle:", e)

In [ ]:
def elegir_tema(respuesta, temas):
    """Acepta la respuesta solo si es uno de los temas válidos; si no, None."""
    r = respuesta.strip().strip(".\"'«»").lower()
    return r if r in temas else None

resp = preguntar_llm(prompt_few_shot("Aguinaldo de Navidad para pensionados IPS", EJEMPLOS, TEMAS),
                     respaldo=RESPALDO_TEMA)
print("respuesta:", repr(resp), "→ tema válido:", elegir_tema(resp, TEMAS))
print("tema oficial en ChileAtiende:", df.loc[df["titulo"] == "Aguinaldo de Navidad para pensionados IPS", "tema"].iloc[0])

## Temperatura y alucinación

La **temperatura** controla cuánto «se arriesga» el modelo al elegir cada palabra: cerca de **0**
elige casi siempre la opción más probable (respuestas estables, repetibles); valores altos
(1 a 2) eligen opciones menos probables (más variedad… y más riesgo de inventar). Para extraer
datos de una ficha quieres temperatura **baja**; para lluvia de ideas, puedes subirla.

Con clave, la celda de abajo pide la misma extracción dos veces con temperatura 0 y dos con 1,8:
compara. Sin clave verás el respaldo — y la lección es la misma: **ninguna temperatura garantiza
la verdad**. Un modelo puede inventar con temperatura 0. Por eso, lo que sigue es **verificar**.

In [ ]:
def preguntar_llm_temperatura(prompt, temperatura, respaldo):
    """Como preguntar_llm, pero fijando la temperatura (0 = estable, 2 = muy variable)."""
    clave = _clave_gemini()
    if not clave:
        return respaldo
    try:
        from google import genai
        from google.genai import types
        cliente = genai.Client(api_key=clave)
        r = cliente.models.generate_content(
            model=MODELO, contents=prompt,
            config=types.GenerateContentConfig(temperature=temperatura))
        return r.text
    except Exception as e:
        print("⚠️ No se pudo llamar al LLM:", e)
        return respaldo

for t in [0.0, 1.8]:
    for intento in [1, 2]:
        r = preguntar_llm_temperatura(prompt_extraccion(invierno, fragmento), t, RESPALDO_EXTRACCION)
        d = parsear_json(r)
        print(f"temperatura {t} · intento {intento}:", d["montos_clp"] if d else "(JSON inválido)")

## 7. Verificar las cifras contra la fuente

La primera defensa contra la alucinación es **automática**: todo monto que el LLM extrajo debe aparecer en el texto fuente. `montos_en_texto` (ya escrita) encuentra los montos con `$` de un texto y los convierte a enteros. Escribe `montos_no_respaldados(montos, fuente)`: la lista de montos extraídos que **no** están en la fuente.

### ✍️ Ejercicio 5 — Escribe `montos_no_respaldados`

In [ ]:
def montos_en_texto(texto):
    """{81257, 231440} a partir de '... $81.257 ... $231.440 ...'"""
    return {int(m.replace(".", "")) for m in re.findall(r"\$\s?(\d{1,3}(?:\.\d{3})+|\d+)", texto)}

def montos_no_respaldados(montos, fuente):
    en_fuente = montos_en_texto(fuente)
    return [m for m in montos if m not in en_fuente]

print("montos en la fuente:", montos_en_texto(fragmento))
print("respaldo:", montos_no_respaldados(parsear_json(RESPALDO_EXTRACCION)["montos_clp"], fragmento))
print("dudosa:  ", montos_no_respaldados(parsear_json(RESPUESTA_DUDOSA)["montos_clp"], fragmento))

In [ ]:
# ── Celda de chequeo · Ejercicio 5 ──────────────────────────────────────────
try:
    assert montos_en_texto(fragmento) == {81257, 231440}
    assert montos_no_respaldados(parsear_json(RESPALDO_EXTRACCION)["montos_clp"], fragmento) == []
    assert montos_no_respaldados(parsear_json(RESPUESTA_DUDOSA)["montos_clp"], fragmento) == [85000]
    assert montos_no_respaldados([250275, 1], pgu["texto"]) == [1]
    print("✅ Ejercicio 5: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'en_fuente = montos_en_texto(fuente); return [m for m in montos if m not in en_fuente].')
    print("   Detalle:", e)

## La verificación humana no se automatiza del todo

Tu verificador atrapó el monto inventado (`85000`). Pero mira los **requisitos** de la
`RESPUESTA_DUDOSA` junto al texto fuente:

In [ ]:
dudosa = parsear_json(RESPUESTA_DUDOSA)
for req in dudosa["requisitos"]:
    print("•", req)
print("\nFUENTE:\n" + fragmento)

«Postular en tu AFP antes del 30 de abril» **no está en la fuente** — y la fuente dice lo
contrario: es un beneficio **no postulable**. Ninguna regla simple lo detecta: hace falta una
persona que lea. Lista de verificación mínima antes de publicar algo producido por un LLM:

1. **Cifras y fechas:** ¿cada monto, edad y fecha aparece en la fuente oficial? (automatizable, ejercicio 5)
2. **Requisitos y condiciones:** ¿cada uno está en la fuente, sin agregados ni contradicciones? (lectura humana)
3. **Omisiones:** ¿falta algo que cambia quién accede al beneficio? (lectura humana)
4. **Datos personales:** ¿el texto enviado al LLM estaba libre de ellos? (automatizable, ejercicio 6)
5. **Responsable:** ¿quién firma? El LLM asiste; la responsabilidad pública es de una persona.

## 8. Guardrail: nada con RUT sale hacia el LLM

Antes de enviar cualquier texto a un servicio externo, revisa que no lleve datos personales. El caso más típico en Chile es el **RUT** (`12.345.678-9`, con o sin puntos, dígito verificador número o `K`). Escribe `es_seguro_enviar(texto)`: `False` si encuentra un RUT, `True` si no. Patrón: `r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b"`.

### ✍️ Ejercicio 6 — Escribe `es_seguro_enviar`

In [ ]:
PATRON_RUT = r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b"

def es_seguro_enviar(texto):
    return re.search(PATRON_RUT, texto) is None

consulta = "Mi RUT es 12.345.678-9 y quiero saber si me corresponde el Bono de Invierno"
print(es_seguro_enviar(pgu["texto"]), es_seguro_enviar(consulta))

In [ ]:
# ── Celda de chequeo · Ejercicio 6 ──────────────────────────────────────────
try:
    assert es_seguro_enviar(pgu["texto"]) == True
    assert es_seguro_enviar(consulta) == False
    assert es_seguro_enviar("RUT 9876543-K, pensionado") == False, "también sin puntos y con K"
    assert all(es_seguro_enviar(t) for t in df["texto"]), "las fichas públicas no traen RUT: deben pasar"
    print("✅ Ejercicio 6: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'return re.search(PATRON_RUT, texto) is None.')
    print("   Detalle:", e)

## Cierre

- Un **LLM** capta significado: resume, extrae y clasifica con pocas instrucciones — tú lo diriges con el **prompt** (rol, tarea, reglas, fuente).
- El prompt define un **contrato**; tu código lo hace cumplir: **parser** tolerante, **validador** de claves y tipos.
- **Few-shot**: unos pocos ejemplos en el prompt reemplazan el entrenamiento — pero la salida igual se **valida** contra la lista de temas.
- La **temperatura** regula la variabilidad, no la verdad: las cifras se **verifican contra la fuente** y los requisitos los **lee una persona**.
- **Nunca** pegues tu clave en el notebook ni envíes datos personales: secretos de Colab y guardrail.

> *En R3-03 (RAG) le darás al LLM las fichas como **fuente** para responder preguntas de la
> ciudadanía citando de dónde sacó cada dato — con la verificación de hoy como red de seguridad.*